# 02. クレンジング

01章で1枚にまとめた36行を、**計算できる形に直す**のがこの章です。

## この章のゴール

```
  raw  36行 (全部文字列)
    ↓   表記を揃える / 欠損を決める / 型を変える / 税を揃える
  clean     34行  使える行
  rejected   2行  使えないので隔離した行 (捨てない)
```

01章の最後に残した「持ち越す宿題」を、上から順に潰していきます。

## 章をまたいで使う名前

この教材では、次の5つの名前を**章をまたいで同じ意味で**使います。
どの段階の表を指しているかが、名前だけで分かるようにするためです。

| 名前 | 中身 | 作る章 |
| --- | --- | --- |
| `raw` | 3系統を積んだだけ。全部文字列 | 01章 |
| `clean` | 表記と型を整え、使えない行を除いたもの | **02章(この章)** |
| `rejected` | 使えないので隔離した行。捨てていない | **02章(この章)** |
| `fact` | `clean` にマスタを結合したもの。除外はまだしない | 03章 |
| `target` | `fact` から集計対象外を除いたもの | 04章 |

## この章の進め方

01章と同じです。**考え方を読む → セルを実行する → 出力の読み方を知る → 1問だけ書いてみる。**

「書いてみる」は、分からなければすぐ下の**答え**を開いて写してかまいません。

> この章では**行を1つも捨てません。** 使えない行は `rejected` に分けて取っておきます。
> なぜそうするかは、9節で説明します。

---
## 0. 前章までのまとめ

**次のセルは01章の答えです。読み飛ばして実行してかまいません。**

各章はここから始められるようにしてあるので、01章をやっていなくても先に進めます。

In [ ]:
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

SOURCES = [
    {
        "path": "/data/sales_2024-04_old.csv",
        "encoding": "cp932",
        "tax_type": "税込",
        "source": "old",
        "rename": {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額": "amount", "備考": "note"},
    },
    {
        "path": "/data/sales_2024-04_new.csv",
        "encoding": "utf-8",
        "tax_type": "税込",
        "source": "new",
        "rename": {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額": "amount", "備考": "note"},
    },
    {
        "path": "/data/sales_2024-04_ec.csv",
        "encoding": "utf-8",
        "tax_type": "税抜",
        "source": "ec",
        "rename": {"受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                   "数量": "qty", "金額税抜": "amount", "ステータス": "note"},
    },
]


def read_one(spec):
    df = pd.read_csv(spec["path"], dtype=str, keep_default_na=False,
                     encoding=spec["encoding"])
    df = df.rename(columns=spec["rename"])
    df = df.assign(tax_type=spec["tax_type"], source=spec["source"])
    return df[COLUMNS]


def load_raw(sources=SOURCES):
    frames = [read_one(s) for s in sources]
    df = pd.concat(frames, ignore_index=True)
    print(f"取り込み: {len(df)}行  内訳 {df['source'].value_counts().to_dict()}")
    return df


raw = load_raw()
raw.head()

---
## 1. 直す順番を先に決める

いきなり手を動かす前に、**どの順で直すか**を決めます。
クレンジングは**順番によって結果が変わる**からです。

読み物のほうに、この順番が書いてあります(読み物 `docs/03 データは汚い` の「正規化の順番」)。

```
1. 読む            ← 01章でやった
2. 全部文字列で持つ  ← 01章でやった
3. Unicode正規化    ← ここから (2節)
4. 空白除去
5. 欠損の判定       (3節)
6. 型変換          (4〜7節)
7. 重複排除        ← 03章
8. 除外            ← 04章
9. 型を固める       ← 05章
```

**なぜ「正規化」が「欠損の判定」より先なのか。** 欠損を表す `-` に、
全角空白が付いて `　-　` と書かれていることがあるからです。

```
先に欠損判定 →  "　-　" は "-" と一致しないので、欠損として拾えない
先に正規化   →  " - " → strip → "-"  → 拾える
```

この章は、上の 3 → 4 → 5 → 6 をこの順でなぞります。
7・8・9 は、それぞれ 03章・04章・05章です。

> 順番の理由を1つずつ覚える必要はありません。
> **「正規化 → 欠損 → 型」の3つの塊**として覚えておけば、たいていの場面で足ります。

---
## 2. 表記を揃える (NFKC)

01章で見つけた汚れのうち、**全角がらみのものは一度に片付きます。**

- 商品CDの `０００１`(全角数字)
- 数量の `２`、金額の `９００`
- 店名の `ミナトストア　渋谷`(全角空白)
- 金額の `￥1,200`(全角の円記号)

これを1つずつ `replace` していくと必ず漏れが出ます。
**Unicode正規化**という、まとめて標準の形に寄せる仕組みを使います。

### 2-1. 1文字ずつ試してみる

`unicodedata.normalize("NFKC", s)` です。まず値1つで確かめます。

In [ ]:
for s in ["０００１", "２", "９００", "ミナトストア　渋谷", "￥1,200"]:
    print(f"{s!r:24} → {unicodedata.normalize('NFKC', s)!r}")

全角数字が半角に、全角空白が半角空白に、`￥` が `¥` になりました。
**4種類の汚れが1つの関数で片付いています。** これが正規化を使う理由です。

> `!r` は `repr()` で表示する指定です。`'ミナトストア 渋谷'` のように
> **クォートが付くので、空白が見えます。** 空白の汚れを見るときは必ず付けてください。
> `print(s)` だけだと、全角空白なのか半角空白なのか区別が付きません。

In [ ]:
# ✍ 書いてみる: 全角の "０００５" を NFKC で正規化した結果を ans に入れてください。

ans = ...   # ここに書く

assert ans == "0005", f"'0005' のはずです: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = unicodedata.normalize("NFKC", "０００５")
```

</details>

### 2-2. NFKC がやりすぎる場合

便利な反面、**残したい表記まで変えてしまう**ことがあります。

In [ ]:
for s in ["㈱", "①", "Ⅲ", "(株)"]:
    print(f"{s!r:8} → {unicodedata.normalize('NFKC', s)!r}")

`㈱` は `(株)` に、`①` は `1` に、`Ⅲ` は `III` に変わります。
**商品名や顧客名に使うと、元の表記が失われます。**

このデータにある `(株)みなとストア 新宿店` は、はじめから半角の括弧なので変わりません。
今回は問題になりませんが、**「NFKC は安全な操作ではない」**とは覚えておいてください。

対処は2つです。

- 正規化する列を限定する(コードや数量にだけかける)
- 正規化前の値も別の列に残しておく

この教材では、**明細の列は全部かけて構わない**と判断しています。
店名も商品名も、あとでマスタに突き合わせて正式名称を取り直すからです(03章)。

### 2-3. 空白を落とす (strip)

正規化の次は空白です。`　`(全角空白)は 2-1 で半角空白になっているので、
あとは `strip()` で前後を落とせば済みます。

In [ ]:
s = "　みなとストア渋谷店　"
print(f"元          {s!r}")
print(f"NFKCだけ     {unicodedata.normalize('NFKC', s)!r}")
print(f"NFKC+strip  {unicodedata.normalize('NFKC', s).strip()!r}")

**NFKC だけでは空白は消えません。** 全角空白が半角空白になるだけです。
`strip()` まで通して、はじめて `'みなとストア渋谷店'` になります。

> 今回の `data/` には、前後に空白が付いた値は**入っていません**。
> それでも `strip()` を入れておくのは、**入っていても気づけないから**です。
> 見えない汚れなので、入っていないことを確認するより、常に落とすほうが安上がりです。

In [ ]:
# ✍ 書いてみる: 全角空白で囲まれた "　１２３　" を、NFKC と strip の両方を通して
#              "123" にしてください。

ans = ...   # ここに書く

assert ans == "123", f"'123' のはずです: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = unicodedata.normalize("NFKC", "　１２３　").strip()
```

</details>

### 2-4. 列全体にかける

値1つで確かめたので、表に適用します。
**1列の全要素に関数をかけるのは `map`** です。

In [ ]:
def norm(s):
    """NFKC正規化して、前後の空白を落とす。"""
    return unicodedata.normalize("NFKC", s).strip()


TEXT_COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount", "note"]

clean = raw.copy()
for c in TEXT_COLUMNS:
    clean[c] = clean[c].map(norm)

clean.head(3)

`raw` を `copy()` してから直しているところに注目してください。
**元の `raw` は壊しません。** あとで「元は何だったか」を見比べたくなるからです。

`tax_type` と `source` を対象から外しているのは、
**この2列は自分で付けた値で、汚れていないと分かっている**からです。

### 何が変わったか、数えて確かめる

In [ ]:
for c in ["shop_name", "item_cd", "qty", "amount"]:
    changed = (raw[c] != clean[c]).sum()
    print(f"{c:10} 変わった行数: {changed:3}   種類 {raw[c].nunique()} → {clean[c].nunique()}")

**`item_cd` は17種類から12種類に減りました。** `０００１` と `0001` が同じものになったからです。

一方 **`shop_name` は8種類のまま**です。ここが大事なところで、

```
ミナトストア　渋谷  →  ミナトストア 渋谷     全角空白は消えた
みなとストア渋谷店  →  みなとストア渋谷店     変わらない
みなと渋谷         →  みなと渋谷            変わらない
```

**NFKC は表記ゆれを直しません。** 全角と半角を揃えるだけです。
「ミナトストア 渋谷」と「みなとストア渋谷店」が同じ店だ、というのは
Unicode の知識ではなく**業務の知識**なので、辞書を使って寄せます。これが03章です。

In [ ]:
# ✍ 書いてみる: clean の qty 列に入っている値の種類を、ソートしたリストで出してください。
#              (ヒント: sorted(...unique()))

ans = ...   # ここに書く

assert ans == ["-", "-1", "1", "2", "3", "4"], ans
print("OK")

<details>
<summary>答え</summary>

```python
ans = sorted(clean["qty"].unique())
```

</details>

全角の `２` が消えて、`-`(欠損)と `-1`(返品)だけが数値以外として残りました。
**残りは「欠損の話」です。** 次の節でやります。

---
## 3. 何を「欠損」と決めるか

01章で、欠損の表し方が**3種類**あることを見つけました。

| 表現 | 出てくる場所 |
| --- | --- |
| `""`(空文字) | `note` のほとんど、旧POSと新POSの備考欄 |
| `-` | 新POSの `qty` |
| `N/A` | 新POSの `amount` |

pandas は `keep_default_na=False` で読んだので、**この3つはどれも普通の文字列**です。
どれを欠損と見なすかは、**自分で決めます**。

### 3-1. 欠損の候補を数える

決める前に、どこにいくつあるかを見ます。

In [ ]:
for token in ["", "-", "N/A"]:
    hit = (clean == token).sum()
    hit = hit[hit > 0]
    print(f"{token!r:6} → {hit.to_dict()}")

- 空文字は `note` に24件。これは「備考なし」であって、異常ではありません
- `-` は `qty` に1件
- `N/A` は `amount` に1件

**3つとも欠損として扱って構いません。** `note` の空文字を欠損にしても、
「備考が無い」という意味は変わらないからです。

> 列ごとに欠損の定義を変えたくなることもあります(`0` を欠損と見なす、など)。
> そのときは列ごとに処理を分けます。**今回は全列同じでよい**、と判断しました。
> こういう判断を1つずつ言葉にしておくと、あとで読み返したときに追えます。

### 3-2. `pd.NA` に置き換える

`replace` に**リストを渡すと、まとめて置き換えられます**。

In [ ]:
clean = clean.replace(["", "-", "N/A"], pd.NA)

print(clean.isna().sum())

01章では `isna()` が**全部ゼロ**でした。いま数字が入ったのは、
**欠損として扱うことにした**からです。データは何も変わっていません。変えたのは扱いです。

`qty` が1件、`amount` が1件。この2行が、9節で隔離することになる行です。

> `pd.NA` ではなく `None` や `np.nan` を使うこともできますが、
> **`pd.NA` に統一しておくのが無難です。** `np.nan` は float なので、
> 文字列の列に混ぜると型が揺れます。

In [ ]:
# ✍ 書いてみる: 欠損が1件でもある行だけを取り出してください。
#              (ヒント: 行方向の any は df.isna().any(axis=1))

ans = ...   # ここに書く

assert len(ans) == 25, f"25行のはずです: {len(ans)}"
assert ans["source"].value_counts().to_dict() == {"old": 13, "new": 12}, \
    ans["source"].value_counts().to_dict()
print("OK")

<details>
<summary>答え</summary>

```python
ans = clean[clean.isna().any(axis=1)]
```

</details>

**25行**もありました。ほとんどが `note` の空欄です。

ここで「26行も欠損がある、大変だ」と慌てないでください。
**欠損のある列がどこかが問題**で、行数そのものには意味がありません。
`note` が空でも売上の集計には影響しません。

EC(`ec`)が1行も出てこないのは、`ステータス` に必ず `確定` か `返品` が入っているからです。

---
## 4. 商品コードの桁を揃える

NFKC で `０００１` は `0001` になりました。それでも、まだ揃っていないものがあります。

In [ ]:
print(sorted(clean["item_cd"].unique()))

`1` と `0001` が別々に居ます。**EC だけ先頭のゼロが落ちている**からです。

これは 01章 の 4-3 で見つけていた宿題です。
原因はたいてい、**送信元のどこかで数値として扱われた**ことです。
`0001` を数値にすると `1` になり、文字列に戻しても `0001` には戻りません。

> だから読み物 `docs/03`は「**IDは文字列で扱う**」と書いています。
> 自分が同じ事故を起こさないために、01章で `dtype=str` を指定していました。

受け取る側では、**桁数を決めてゼロ埋めし直します**。
`str.zfill(4)` は、4桁になるまで左を `0` で埋めます。

In [ ]:
clean["item_cd"] = clean["item_cd"].str.zfill(4)

print(sorted(clean["item_cd"].unique()))

**7種類に収束しました。** `0001`〜`0006` と `0099` です。

`0099` は商品マスタに無いコードです。ここではまだ触りません。03章で扱います。

> `zfill` は**もともと4桁のものには何もしません。** `"0001".zfill(4)` は `"0001"` のままです。
> だから「ECだけ」と条件を分ける必要がありません。
> **系統ごとに分岐せずに済む書き方**を選ぶと、ファイルが増えたときに壊れにくくなります。

In [ ]:
# ✍ 書いてみる: ゼロ埋めのあとの item_cd を、値ごとの件数で出してください。

ans = ...   # ここに書く

assert len(ans) == 7, f"7種類のはずです: {len(ans)}"
assert ans["0001"] == 11, f"0001 は11件のはずです: {ans['0001']}"
assert ans.sum() == 36
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = clean["item_cd"].value_counts()
```

</details>

---
## 5. 金額から記号を落とす

01章で「数値にできない値」を洗い出したとき、3種類出てきました。

| 値 | いまの状態 |
| --- | --- |
| `９００`(全角) | **2節の NFKC で解決済み** |
| `￥1,200` | `¥1,200` になった。**記号とカンマがまだ残っている** |
| `N/A` | **3節で `pd.NA` にした** |

残っているのは真ん中だけです。

In [ ]:
print(clean.loc[clean["amount"].str.contains("¥", na=False), "amount"].tolist())

2件です。`￥`(全角)ではなく `¥`(半角)になっていることを確認してください。
**NFKC を通したあとなので、探すのは半角のほう**です。

> ここは間違えやすいところです。生データを見て `￥` で検索を書くと、
> **正規化のあとでは1件も引っかかりません。** 「いまどの形になっているか」を
> 毎回確かめてから条件を書いてください。

`str.replace` に正規表現で「`¥` かカンマ」を渡して、まとめて落とします。

In [ ]:
clean["amount"] = clean["amount"].str.replace(r"[¥,]", "", regex=True)

print(clean.loc[[16, 18], ["amount", "source"]])

`1200` と `1650` になりました。

> `[¥,]` は「`¥` かカンマのどちらか1文字」という意味です。
> `regex=True` を付け忘れると、**`"¥,"` という2文字の並びを探しに行く**ので何も置き換わりません。
> 置換したのに変わらないときは、まずここを疑ってください。

In [ ]:
# ✍ 書いてみる: 金額に「数字以外の文字」がまだ残っている行が無いことを確かめてください。
#              (ヒント: 記号や文字にマッチする正規表現を str.contains で。欠損は na=False で外す)

ans = ...   # ここに書く

assert ans == 0, f"0件のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = clean["amount"].str.contains(r"[^0-9-]", na=False).sum()
```

</details>

`[^0-9-]` は「数字でもマイナスでもない文字」です。0件なので、
**金額はもう数値にできる形**になりました。実際に変換するのは7節です。

---
## 6. 日付を1つの型にする

01章で、日付が**3書式**あることを確認しました。

| source | 例 |
| --- | --- |
| `old` | `2024年4月1日` |
| `new` | `2024/4/1` |
| `ec` | `2024-04-02` |

### 6-1. まず「1つの書式では無理」を確かめる

`pd.to_datetime` に書式を1つ渡してみます。

In [ ]:
one = pd.to_datetime(clean["sale_date"], format="%Y/%m/%d", errors="coerce")

print("変換できた:", one.notna().sum())
print("できなかった:", one.isna().sum())

12件しか変換できません。`new` のぶんだけです。

`errors="coerce"` は、**変換できなかったものを `NaT`(日付の欠損)にする**指定です。
これを付けないと、最初の1件でエラーになって止まります。
**まず全体を通して「何件だめか」を知りたい**ので、調査のときは `coerce` を使います。

> `format` を省略すると pandas が推測してくれますが、
> **推測は使わないほうが安全です。** `01/02/2024` が1月2日なのか2月1日なのか、
> pandas には分かりません。書式が分かっているなら、必ず書いてください。

### 6-2. 書式ごとに変換して、重ねる

考え方はこうです。

```
書式Aで変換    →  Aの行だけ埋まる。残りは NaT
書式Bで変換    →  Bの行だけ埋まる
   ↓ NaT のところに、Bの結果を流し込む (fillna)
書式Cで変換    →  同じように流し込む
   ↓
全部埋まる
```

**`fillna` で重ねていく**のがこつです。

In [ ]:
FORMATS = ["%Y/%m/%d", "%Y年%m月%d日", "%Y-%m-%d"]

d = pd.to_datetime(clean["sale_date"], format=FORMATS[0], errors="coerce")
for fmt in FORMATS[1:]:
    d = d.fillna(pd.to_datetime(clean["sale_date"], format=fmt, errors="coerce"))
    print(f"{fmt:14} まで適用   埋まった: {d.notna().sum():2} / {len(d)}")

clean["sale_date"] = d
clean["sale_date"].head(3)

12 → 26 → 36 と埋まっていきました。**全36行が日付型になりました。**

```
dtype: datetime64[ns]
```

型が `object`(文字列)から変わっているのを確認してください。これで
`sale_date.dt.day` や大小比較が使えるようになります。

> pandas 2.x には `format="mixed"` という指定もあります。行ごとに書式を推測してくれますが、
> `2024年4月1日` のような書式は解釈できませんし、推測である点も変わりません。
> **書式が分かっているなら、並べて書くほうが確実です。**

In [ ]:
# ✍ 書いてみる: 日付が入っている範囲 (最小と最大) を出してください。

ans = ...   # ここに書く

assert ans == (pd.Timestamp("2024-04-01"), pd.Timestamp("2024-04-10")), ans
print("OK")

<details>
<summary>答え</summary>

```python
ans = (clean["sale_date"].min(), clean["sale_date"].max())
```

</details>

4/1 から 4/10 に収まっています。**4月のファイルなのだから4月の日付しか無いはず**、
という当たり前の確認です。

こういう確認を1つ入れておくと、「5月分のファイルが4月のフォルダに混ざっていた」
のような事故に気づけます。**期待した範囲に入っているか**は、日付を変換したら毎回見てください。

---
## 7. 数量と金額を数値にする

記号は落としました。あとは型を変えるだけです。

In [ ]:
clean["qty"] = pd.to_numeric(clean["qty"], errors="coerce").astype("Int64")
clean["amount"] = pd.to_numeric(clean["amount"], errors="coerce").astype("Int64")

print(clean.dtypes)

### `Int64` と `int64` は違います

大文字の `Int64` は **欠損を持てる整数型**です。小文字の `int64` では持てません。

In [ ]:
s = pd.Series(["1", "2", None])

print("Int64 :", pd.to_numeric(s).astype("Int64").tolist())
try:
    pd.to_numeric(s).astype("int64")
except Exception as e:
    print("int64 :", f"{type(e).__name__}: {e}")

`int64` は欠損があると変換できません。
**逃げ道として float にすると、今度は `2` が `2.0` になります**(金額が小数になるのは気持ち悪い)。

いま `qty` と `amount` には欠損が1件ずつ残っているので、`Int64` を使います。
9節でその行を隔離したあとは `int64` にできますが、
**05章で型を固めるまでは `Int64` のまま**運びます。

In [ ]:
# ✍ 書いてみる: 数値にできず NaN になった行が、3節で欠損にした2件だけであることを
#              確かめてください。qty と amount の欠損数をタプルで返します。

ans = ...   # ここに書く

assert ans == (1, 1), f"(1, 1) のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (clean["qty"].isna().sum(), clean["amount"].isna().sum())
```

</details>

**1件ずつのままです。** 新しく変換に失敗した行はありません。

これが確認できて、はじめて「2節から6節までの整形が効いた」と言えます。
**型を変えたら、欠損が増えていないかを必ず見てください。**
`errors="coerce"` は失敗を黙って `NaN` にするので、確認しないと取りこぼしに気づけません。

---
## 8. 税込と税抜を揃える

ここがこの章でいちばん大事なところです。

01章で `tax_type` 列を付けました。**店舗POSは税込、ECは税抜**です。
このまま合計すると、EC のぶんだけ約10%小さい金額が混ざります。
**エラーは出ません。** 数字が静かに狂うだけです。

In [ ]:
print(clean.groupby("tax_type")["amount"].agg(["count", "sum"]))

税抜が10行、税込が25行。**この2つを足しても意味のある数字になりません。**

> 税込は26行あるはずなのに `count` が25なのは、`count` が**欠損を除いて数える**からです。
> 3節で `N/A` を欠損にした1行が引かれています。
> **行数を知りたいときは `len`、値が入っている数を知りたいときは `count`** です。
> ここを混同すると、原因不明の1件差を追いかけることになります。

**どちらに揃えるかを決めます。** ここでは税込に揃えます。
売上サマリを見る人が知りたいのは、お客さんが払った金額だからです。

### 8-1. 税込の列を作る

**元の `amount` は残したまま、`amount_incl` を足します。**
上書きしないのは、あとで「元はいくらだったか」を追えるようにするためです。

In [ ]:
TAX_RATE = 1.1

is_excl = clean["tax_type"] == "税抜"
converted = (clean["amount"] * TAX_RATE).round()

clean["amount_incl"] = clean["amount"].where(~is_excl, converted).astype("Int64")

clean.loc[clean["source"] == "ec", ["item_cd", "qty", "amount", "amount_incl", "note"]]

`where` は「条件が **True のところは元のまま**、False のところを置き換える」です。
`~is_excl`(税抜でない = 税込)の行はそのまま、税抜の行だけ `converted` に差し替わります。

> `if` で書きたくなりますが、`if` は Series 全体には使えません
> (`The truth value of a Series is ambiguous` というエラーになります)。
> **行ごとの分岐は `where` か `np.where`** で書きます。

返品の2行(`-500` と `-364`)も、マイナスのまま `-550` と `-400` になっています。
**マイナスを特別扱いする必要はありません。** 掛け算はそのまま通ります。

In [ ]:
# ✍ 書いてみる: amount と amount_incl が違う行が何行あるか数えてください。

ans = ...   # ここに書く

assert ans == 10, f"10行のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (clean["amount"] != clean["amount_incl"]).sum()
```

</details>

**EC の10行だけ**が変わりました。店舗POSの26行は触っていません。狙いどおりです。

### 8-2. 掛けて戻した金額は、元には戻りません

ここで1つ、気づいておいてほしいことがあります。

同じ商品が、店舗POSでは税込で、EC では税抜で届いています。
**両方を税込にしたら、同じ値段になるはずです。** 確かめてみます。

In [ ]:
items = pd.read_csv("/data/items.csv", dtype=str, keep_default_na=False)
price = items.set_index("item_cd")[["item_name", "list_price"]]

ec = clean[clean["source"] == "ec"].join(price, on="item_cd")
ec = ec.assign(
    定価から計算=lambda d: d["list_price"].astype(int) * d["qty"],
    差=lambda d: d["amount_incl"] - d["list_price"].astype(int) * d["qty"],
)
ec[["item_name", "qty", "amount", "amount_incl", "定価から計算", "差"]]

**3行で1円ずれています。**

| 商品 | 数量 | 税抜 | ×1.1 して四捨五入 | 定価×数量 |
| --- | --- | --- | --- | --- |
| ケーキ | 2 | 1090 | **1199** | 1200 |
| 紅茶 | 2 | 728 | **801** | 800 |
| スープ | 2 | 872 | **959** | 960 |

理由は単純です。**税抜の金額そのものが、すでに丸められている**からです。

```
ケーキ 1個 600円(税込)
  → 税抜は 600 ÷ 1.1 = 545.45...  → 545 に丸めて EC に記録された
  → 2個で 1090
  → 1090 × 1.1 = 1199.0    600 × 2 = 1200 には戻らない
```

**丸めた数を掛け戻しても、元には戻りません。** これは計算方法の間違いではなく、
桁を落とした時点で情報が失われている、というだけのことです。

### だからどうするか

覚えておくことは3つです。

- **税抜から作った税込は「復元値」であって、正しい税込ではありません。**
  1円単位を保証しないといけない用途(請求・会計)には使えません
- **元の `amount` を残しておきます。** ずれが問題になったとき、
  `amount_incl` から逆算するのではなく、元の値に戻って調べられます
- **どちらの数字を採るかは業務が決めることです。** 迷ったら、
  「この数字を見るのは誰で、何に使うのか」に戻ります

この教材の売上サマリは**傾向を見るためのもの**なので、1円のずれは許容して先に進みます。
そのかわり、**ずれていることを知らないまま進まない**、というのがこの節の主旨です。

> 実務では、税率が複数(8%と10%)あったり、税率の適用日が途中で変わったりします。
> そうなると `TAX_RATE = 1.1` では足りず、**税率マスタ**を持つことになります。
> 考え方は同じで、「掛け算の前に、どの税率かを列として持たせる」ことになります。

In [ ]:
# ✍ 書いてみる: 上の ec の中で、差がゼロでない行数を数えてください。

ans = ...   # ここに書く

assert ans == 3, f"3行のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (ec["差"] != 0).sum()
```

</details>

---
## 9. 使えない行を、捨てずに隔離する

`qty` と `amount` に欠損が1件ずつ残っています。この2行は集計に使えません。

**ここで `dropna()` を書きたくなりますが、その前に考えます。**

```
落とす            →  36行が34行になる。何が落ちたかは残らない
隔離する (取っておく)  →  34行 + 2行。あとで「なぜ落ちたか」を見られる
```

読み物 `docs/03` のチェックリストには、こう書いてあります。

> - [ ] 落とした行を数えているか。黙って捨てていないか

**除外そのものは必要な処理です。** 問題は、黙ってやることです。
除外率がいつもの1%から10%に跳ねたときに、それに気づけるかどうかが分かれ目になります。

### 9-1. 使える条件を書く

まず「使える行」の条件を書きます。**否定形より肯定形のほうが読みやすい**です。

In [ ]:
ok = clean["sale_date"].notna() & clean["qty"].notna() & clean["amount"].notna()

rejected = clean[~ok].copy()
clean = clean[ok].reset_index(drop=True)

print(f"使える行: {len(clean)}")
print(f"隔離した行: {len(rejected)}")
rejected[["sale_date", "shop_name", "item_cd", "qty", "amount", "note", "source"]]

隔離されたのはこの2行です。

| 日付 | 店 | 商品 | 何が欠けているか | 備考 |
| --- | --- | --- | --- | --- |
| 4/9 | 横浜店 | 0003 | **数量** | `数量不明` |
| 4/10 | 渋谷店 | 0001 | **金額** | (なし) |

`備考` に `数量不明` と書いてあるほうは、**送信元も欠けていることを分かっている**ようです。
もう1件は何も書かれていません。

> `~ok` の `~` は「否定」です。`not ok` とは書けません
> (Series には `not` が使えないので、`~` を使います)。

> `reset_index(drop=True)` を付けているのは、行を抜いたあとの index が
> `0,1,2,4,5,...` と飛ぶからです。**飛んだままでも動きますが、
> あとで `iloc` と混同する事故のもと**になるので、振り直しておきます。

In [ ]:
# ✍ 書いてみる: 隔離した行の割合 (%) を、小数第1位まで求めてください。

ans = ...   # ここに書く

assert ans == 5.6, f"5.6 のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = round(len(rejected) / (len(clean) + len(rejected)) * 100, 1)
```

</details>

**5.6%** です。この数字を毎回記録しておくのが、docs/03 の言う「数えておく」です。

次に同じパイプラインを流したとき、これが 30% になっていたら、
**上流で何かが変わっています。** 数字を持っていなければ、その変化には気づけません。

### 隔離した行は、このあとどうなるか

**05章で戻ってきます。** `data/sales_2024-04_fix.csv` を開いてみてください。

In [ ]:
print(open("/data/sales_2024-04_fix.csv", encoding="utf-8").read())

```
みなとストア渋谷店,0001,2,900,2024/4/10,訂正    ← 隔離した「金額なし」の行
みなとストア横浜店,0003,2,800,2024/4/9,訂正     ← 隔離した「数量なし」の行
```

**隔離した2行の訂正が、あとから届きます。** 日付も店も商品も一致します。

このとき、02章で `dropna()` して捨てていたとしても困りはしません。
訂正ファイルを足して流し直せばよいだけです。
**ただし「2件落ちていた」ことを知らないと、訂正が届いたことにも気づけません。**

隔離しておく価値は、そこにあります。05章でこの訂正を実際に流します。

---
## 10. この章のまとめ

やったことを**関数1つ**にまとめます。2節から9節までを、この順で並べただけです。

In [ ]:
NA_TOKENS = ["", "-", "N/A"]
TEXT_COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount", "note"]
FORMATS = ["%Y/%m/%d", "%Y年%m月%d日", "%Y-%m-%d"]
TAX_RATE = 1.1


def norm(s):
    """NFKC正規化して、前後の空白を落とす。"""
    return unicodedata.normalize("NFKC", s).strip()


def parse_date(s):
    """3つの書式を順に試して、埋まったものから採る。"""
    d = pd.to_datetime(s, format=FORMATS[0], errors="coerce")
    for fmt in FORMATS[1:]:
        d = d.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return d


def clean_raw(raw):
    """raw を整形して (clean, rejected) に分ける。行は1つも捨てない。"""
    df = raw.copy()

    # 1. 表記を揃える (全角・空白)
    for c in TEXT_COLUMNS:
        df[c] = df[c].map(norm)

    # 2. 欠損を決める
    df = df.replace(NA_TOKENS, pd.NA)

    # 3. 商品コードの桁を揃える
    df["item_cd"] = df["item_cd"].str.zfill(4)

    # 4. 金額から記号を落とす
    df["amount"] = df["amount"].str.replace(r"[¥,]", "", regex=True)

    # 5. 型を変える
    df["sale_date"] = parse_date(df["sale_date"])
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").astype("Int64")

    # 6. 税込に揃える (元の amount は残す)
    is_excl = df["tax_type"] == "税抜"
    df["amount_incl"] = df["amount"].where(
        ~is_excl, (df["amount"] * TAX_RATE).round()).astype("Int64")

    # 7. 使えない行を隔離する (捨てない)
    ok = df["sale_date"].notna() & df["qty"].notna() & df["amount"].notna()
    clean = df[ok].reset_index(drop=True)
    rejected = df[~ok].reset_index(drop=True)

    rate = len(rejected) / len(df) * 100
    print(f"クレンジング: {len(df)}行 → clean {len(clean)}行 / "
          f"rejected {len(rejected)}行 ({rate:.1f}%)")
    return clean, rejected


clean, rejected = clean_raw(raw)
clean.head()

**コメントの番号が、そのまま1節で決めた順番です。**

この形にしておくと、後から読んだ人が「なぜこの順なのか」を追えます。
処理をまとめるときは、**動くようにまとめるのではなく、順番が見えるようにまとめて**ください。

最後に `print` で件数を出しているのも意図があります。
**毎回の実行で件数が記録に残る**ようにしておくと、
異常な回を後から特定できます(読み物 `docs/03` の「件数と合計を記録する」)。

In [ ]:
# ✍ 書いてみる: clean が次の3つを満たすか確かめて、True を ans に入れてください。
#              (1) 34行  (2) sale_date が日付型  (3) qty と amount に欠損が無い

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
ans = bool(
    len(clean) == 34
    and clean["sale_date"].dtype == "datetime64[ns]"
    and clean["qty"].notna().all()
    and clean["amount"].notna().all()
)
```

</details>

### 数字を1つ、覚えておいてください

In [ ]:
print(f"clean          {len(clean)}行")
print(f"qty の合計      {clean['qty'].sum()}")
print(f"amount_incl 合計 {clean['amount_incl'].sum():,} 円")

**26,489円**です。これが「4月に届いた売上の、税込での総額」になります。

03章でマスタと結合しますが、**結合しても、この数字は1円も変わってはいけません。**
変わったら結合の書き方が間違っています。

04章では、ここから**除外**をするので減ります。減った分に説明が付くかどうかが、
そこでの確認になります。**数字を1つ握って次の章に行く**、というのがこつです。

---
## この章で分かったこと

| | |
| --- | --- |
| 順番 | 「正規化 → 欠損 → 型」の順。逆にすると空白付きの欠損を拾えない |
| NFKC | 全角数字・全角空白・全角記号がまとめて片付く。**表記ゆれは直らない** |
| NFKC の副作用 | `㈱`→`(株)`、`①`→`1`。残したい表記がある列にはかけない |
| 欠損 | 何を欠損とするかは**自分で決める**。決めたら `pd.NA` に統一する |
| ID | 桁を決めて `zfill`。もともと合っている値には何も起きない |
| 日付 | 書式ごとに `to_datetime` して `fillna` で重ねる。`format` は省略しない |
| 数値 | 欠損が残るうちは `Int64`(大文字)。変換後に**欠損が増えていないか見る** |
| 税 | 掛け戻した金額は元に戻らない。**元の値を残す** |
| 除外 | 捨てずに隔離して、**割合を記録する** |

## 次の章に持ち越す宿題

01章のチェックリストは、これで片付きました。

- [x] 欠損の表し方が3種類(空文字 / `-` / `N/A`)
- [x] 全角数字(`０００１` `２` `９００`)
- [x] 金額の `￥` とカンマ
- [x] 税込と税抜が混ざっている
- [x] 日付が3書式
- [x] 商品コードの先頭ゼロが落ちている系統がある
- [ ] **店名の表記ゆれ** ← NFKC では直らなかった。03章で辞書を使う

そして、03章で新しく扱うことが3つあります。

- [ ] 店名を `shop_cd` に寄せる(名寄せ)
- [ ] マスタに無い商品(`0099`)と、閉店した店(大宮店)を見つける
- [ ] **01章で保留した重複チェック**をやり直す(表記が揃ったので、今度は意味を持ちます)

次: `03-join.ipynb`